# 🎓 TRAVER + McMiner: Misconception-Aware Coding Tutor
## Running on Google Colab Pro (T4/V100 GPU)

**Architecture:**
- **Tutor**: Llama-3.1-70B-Instruct via HuggingFace Inference API (0 GB GPU)
- **Student Simulator**: Mixtral-8x7B-Instruct via HuggingFace Inference API (0 GB GPU)
- **Verifier**: Pre-trained Verifier-7B loaded locally in 4-bit (~5 GB GPU)
- **McMiner**: Gemini API-based misconception detection (0 GB GPU)

**Phases:**
1. Run baseline TRAVER with pre-trained verifier
2. Run McMiner on student code to detect misconceptions
3. Inject misconceptions into tutor prompts and re-run
4. Evaluate: compare baseline vs misconception-aware TRAVER

**Prerequisites:** Add these keys in Colab → 🔑 Secrets (left sidebar):
- `HF_TOKEN` — your HuggingFace token
- `GOOGLE_API_KEY` — your Gemini API key

---
## 0. GPU Check & Configuration

In [ ]:
# Check GPU (warns if no GPU attached - change runtime to T4 if needed)
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

# Mount Google Drive (safe to re-run: skips if already mounted)
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Google Drive mounted.")
else:
    print("✅ Google Drive already mounted.")

In [ ]:
# ===== LOAD API KEYS FROM COLAB SECRETS =====
# Go to the 🔑 icon in the left sidebar → add HF_TOKEN and GOOGLE_API_KEY
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# ===== MODEL CONFIGURATION =====
TUTOR_MODEL_ID = "meta-llama/Llama-3.1-70B-Instruct"
# Loaded locally on Colab GPU in 4-bit (pre-2024 cutoff, no data contamination)
STUDENT_MODEL_ID = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
# HF Inference Providers: single base URL, model goes in request body
HF_API_BASE = "https://router.huggingface.co/v1"

# ===== STORAGE =====
DRIVE_DIR = "/content/drive/MyDrive/Coding-Tutor-Colab"
WORK_DIR = "/content/Coding-Tutor"
MODEL_DIR = f"{DRIVE_DIR}/models"
DATA_DIR = f"{DRIVE_DIR}/data"

# ===== PIPELINE SETTINGS =====
STUDENT_LEVELS = ["low_level", "med_level", "high_level"]
TUTOR_NUM_RESPONSES = 5  # Reduced from 10 for API rate limits

# ===== DATASET IDS =====
TUTOR_AGENTS_DATASET = "nlpscu/Tutor-Agents"
MCMINER_DATASET = "nlpscu/MCminer"

# Validate
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets (🔑 sidebar)!"
assert GOOGLE_API_KEY, "Add GOOGLE_API_KEY to Colab Secrets (🔑 sidebar)!"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Configuration loaded from Colab Secrets.")

---
## 1. Install Dependencies & Clone Repos

In [ ]:
# Clone Coding-Tutor repo
if not os.path.exists("/content/Coding-Tutor"):
    !git clone https://github.com/iwangjian/Coding-Tutor.git /content/Coding-Tutor
else:
    print("✅ Coding-Tutor already exists.")

# Clone McMiner repo
if not os.path.exists("/content/mcminer"):
    !git clone https://github.com/taisazero/mcminer.git /content/mcminer
else:
    print("✅ mcminer already exists.")

In [ ]:
# Install dependencies
# NOTE: torch is pre-installed on Colab — do NOT reinstall it or pin a version.
# Pinning transformers==4.44.2 also hardcodes torch==2.4.0 which conflicts
# with Colab's torch 2.10.0+cu128. We let pip resolve versions freely.

# Batch 1: Core ML libraries (compatible with Colab's existing torch)
!pip install -q bitsandbytes peft accelerate safetensors

# Batch 2: Transformers + tokenizers (no torch version pinned)
!pip install -q "transformers>=4.44,<4.48" tiktoken sentencepiece protobuf

# Batch 3: API clients
!pip install -q openai huggingface_hub tenacity google-generativeai

# Batch 4: Utilities
!pip install -q tqdm python-dotenv

# Verify torch version is still the Colab-native one
import torch
print(f"✅ All dependencies installed.")
print(f"✅ torch version: {torch.__version__} (should be 2.10.x)")

---
## 2. Download Datasets & Pre-trained Models
Saved to Google Drive — no re-downloading on session restart.

In [ ]:
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

# --- Download Tutor-Agents dataset ---
tutor_data_dir = f"{DATA_DIR}/Tutor-Agents"
if not os.path.exists(tutor_data_dir):
    print("⬇️  Downloading Tutor-Agents dataset...")
    snapshot_download(
        TUTOR_AGENTS_DATASET, repo_type="dataset",
        local_dir=tutor_data_dir, token=HF_TOKEN)
    print("✅ Tutor-Agents dataset downloaded.")
else:
    print("✅ Tutor-Agents dataset already on Drive.")

# --- Download MCminer dataset ---
mcminer_data_dir = f"{DATA_DIR}/MCminer"
if not os.path.exists(mcminer_data_dir):
    print("⬇️  Downloading MCminer dataset...")
    snapshot_download(
        MCMINER_DATASET, repo_type="dataset",
        local_dir=mcminer_data_dir, token=HF_TOKEN)
    print("✅ MCminer dataset downloaded.")
else:
    print("✅ MCminer dataset already on Drive.")

print(f"\n📦 Dataset storage:")
!du -sh {DATA_DIR}/*
# --- Download Student Model locally to Drive ---
student_dir = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
if not os.path.exists(student_dir) or len(os.listdir(student_dir)) < 5:
    print(f"Downloading Student Model to {student_dir}...")
    snapshot_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", local_dir=student_dir, ignore_patterns=["*.pth", "*.h5"])
else:
    print("Student Model already exists in Drive.")


In [ ]:
# --- Download pre-trained Verifier-7B (5 shards, ~1.5 GB) ---
verifier_dir = f"{MODEL_DIR}/Verifier-7B"
if not os.path.exists(f"{verifier_dir}/part0"):
    print("⬇️  Downloading Verifier-7B checkpoints...")
    snapshot_download("jwanglvy/Verifier-7B", local_dir=verifier_dir, token=HF_TOKEN)
    print("✅ Verifier-7B downloaded.")
else:
    print("✅ Verifier-7B already on Drive.")

# --- Download Mistral-7B base model (verifier architecture) ---
mistral_dir = f"{MODEL_DIR}/Mistral-7B-v0.1"
if not os.path.exists(f"{mistral_dir}/config.json"):
    print("⬇️  Downloading Mistral-7B-v0.1...")
    snapshot_download("mistralai/Mistral-7B-v0.1", local_dir=mistral_dir, token=HF_TOKEN)
    print("✅ Mistral-7B-v0.1 downloaded.")
else:
    print("✅ Mistral-7B-v0.1 already on Drive.")

print(f"\n📦 Models:")
!du -sh {MODEL_DIR}/*
# --- Download Student Model locally to Drive ---
student_dir = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
if not os.path.exists(student_dir) or len(os.listdir(student_dir)) < 5:
    print(f"Downloading Student Model to {student_dir}...")
    snapshot_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", local_dir=student_dir, ignore_patterns=["*.pth", "*.h5"])
else:
    print("Student Model already exists in Drive.")


---
## 3. Set Up Working Directory

In [ ]:
import shutil

# Symlink output directory to Drive for persistence
output_link = f"{WORK_DIR}/output"
drive_output = f"{DRIVE_DIR}/output"
os.makedirs(drive_output, exist_ok=True)

if os.path.islink(output_link):
    os.unlink(output_link)
elif os.path.isdir(output_link):
    !cp -rn {output_link}/* {drive_output}/ 2>/dev/null; rm -rf {output_link}
os.symlink(drive_output, output_link)

# Copy MCminer data files into mcminer repo
for fname in ["misconception_bank.json", "problems_processed.json"]:
    src = f"{DATA_DIR}/MCminer/{fname}"
    dst = f"/content/mcminer/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)

# Copy corrupted_codes_best
src_dir = f"{DATA_DIR}/MCminer/corrupted_codes_best"
dst_dir = f"/content/mcminer/corrupted_codes_best"
if os.path.exists(src_dir) and not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)

print("✅ Working directories ready.")
print(f"  Output → {drive_output}")

---
## 4. Patch VLLMChat for HuggingFace Inference API

The original code expects local vLLM servers. We patch `VLLMChat` to:
- Use HF Inference API's OpenAI-compatible endpoint
- Handle `n > 1` by looping (HF doesn't support multi-completion)
- Add retry logic with exponential backoff

In [ ]:
import sys, base64, os
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/traver")

# ── 1. Patch VLLMChat: HF API for tutor, LOCAL 4-bit for student ──
PATCH_FILE = f"{WORK_DIR}/traver/chatarena/backends/openai_vllm.py"
_patch_b64 = "IyBDT0xBQl9QQVRDSEVEIC0gSEYgQVBJIGZvciB0dXRvciArIExPQ0FMIDQtYml0IGZvciBzdHVkZW50CmZyb20gdHlwaW5nIGltcG9ydCBMaXN0CmltcG9ydCBvcywgcmUsIHRpbWUKZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQpmcm9tIC5iYXNlIGltcG9ydCBJbnRlbGxpZ2VuY2VCYWNrZW5kCmZyb20gLi5tZXNzYWdlIGltcG9ydCBNZXNzYWdlLCBTWVNURU1fTkFNRQoKRU5EX09GX01FU1NBR0UgPSAiPEVPUz4iCgpfTE9DQUxfTU9ERUxfQ0FDSEUgPSB7fQoKZGVmIF9sb2FkX2xvY2FsX21vZGVsKG1vZGVsX2lkLCBoZl90b2tlbik6CiAgICBpZiBtb2RlbF9pZCBpbiBfTE9DQUxfTU9ERUxfQ0FDSEU6CiAgICAgICAgcHJpbnQoZiIgIFtWTExNQ2hhdF0gUmV1c2luZyBjYWNoZWQgbG9jYWwgbW9kZWw6IHttb2RlbF9pZH0iKQogICAgICAgIHJldHVybiBfTE9DQUxfTU9ERUxfQ0FDSEVbbW9kZWxfaWRdCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplciwgQml0c0FuZEJ5dGVzQ29uZmlnCiAgICBwcmludChmIiAgW1ZMTE1DaGF0XSBMb2FkaW5nIGxvY2FsIG1vZGVsIGluIDQtYml0OiB7bW9kZWxfaWR9IikKICAgIGJuYl9jb25maWcgPSBCaXRzQW5kQnl0ZXNDb25maWcoCiAgICAgICAgbG9hZF9pbl80Yml0PVRydWUsCiAgICAgICAgYm5iXzRiaXRfcXVhbnRfdHlwZT0ibmY0IiwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsCiAgICApCiAgICB0b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdG9rZW49aGZfdG9rZW4sIHRydXN0X3JlbW90ZV9jb2RlPVRydWUpCiAgICBpZiB0b2tlbml6ZXIucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiA9IHRva2VuaXplci5lb3NfdG9rZW4KICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1vZGVsX2lkLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iX2NvbmZpZywKICAgICAgICBkZXZpY2VfbWFwPSJhdXRvIiwKICAgICAgICB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIHRva2VuPWhmX3Rva2VuLAogICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgbG93X2NwdV9tZW1fdXNhZ2U9VHJ1ZSwKICAgICkKICAgIG1vZGVsLmV2YWwoKQogICAgcHJpbnQoZiIgIFtWTExNQ2hhdF0gTG9jYWwgbW9kZWwgcmVhZHkiKQogICAgX0xPQ0FMX01PREVMX0NBQ0hFW21vZGVsX2lkXSA9IChtb2RlbCwgdG9rZW5pemVyKQogICAgcmV0dXJuIG1vZGVsLCB0b2tlbml6ZXIKCmNsYXNzIFZMTE1DaGF0KEludGVsbGlnZW5jZUJhY2tlbmQpOgogICAgc3RhdGVmdWwgPSBGYWxzZQogICAgdHlwZV9uYW1lID0gInZsbG0tY2hhdCIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgdmxsbV9hcGlfa2V5LCB2bGxtX2VuZHBvaW50LCBtb2RlbF9uYW1lX29yX3BhdGgsCiAgICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9MC43NSwgdG9wX3A9MC45NSwgbWF4X3Rva2Vucz01MDAsCiAgICAgICAgICAgICAgICAgbWF4X2xhdGVzdF9tZXNzYWdlcz0tMSwgKiprd2FyZ3MpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18obW9kZWxfbmFtZV9vcl9wYXRoPW1vZGVsX25hbWVfb3JfcGF0aCwKICAgICAgICAgICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlLCB0b3BfcD10b3BfcCwKICAgICAgICAgICAgICAgICAgICAgICAgIG1heF90b2tlbnM9bWF4X3Rva2VucywKICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9sYXRlc3RfbWVzc2FnZXM9bWF4X2xhdGVzdF9tZXNzYWdlcywgKiprd2FyZ3MpCiAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsX25hbWVfb3JfcGF0aAogICAgICAgIHNlbGYudGVtcGVyYXR1cmUgPSB0ZW1wZXJhdHVyZQogICAgICAgIHNlbGYudG9wX3AgPSB0b3BfcAogICAgICAgIHNlbGYubWF4X3Rva2VucyA9IG1heF90b2tlbnMKICAgICAgICBzZWxmLm1heF9sYXRlc3RfbWVzc2FnZXMgPSBtYXhfbGF0ZXN0X21lc3NhZ2VzCiAgICAgICAgc2VsZi5faXNfbG9jYWwgPSAodmxsbV9lbmRwb2ludCA9PSAibG9jYWwiKQogICAgICAgIGlmIHNlbGYuX2lzX2xvY2FsOgogICAgICAgICAgICBzZWxmLmxvY2FsX21vZGVsLCBzZWxmLmxvY2FsX3Rva2VuaXplciA9IF9sb2FkX2xvY2FsX21vZGVsKG1vZGVsX25hbWVfb3JfcGF0aCwgdmxsbV9hcGlfa2V5KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuY2xpZW50ID0gT3BlbkFJKGFwaV9rZXk9dmxsbV9hcGlfa2V5LCBiYXNlX3VybD12bGxtX2VuZHBvaW50KQoKICAgIGRlZiBfbG9jYWxfY2FsbChzZWxmLCBtZXNzYWdlcyk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcm9tcHQgPSBzZWxmLmxvY2FsX3Rva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICAgICAgbWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXJ0cyA9IFtdCiAgICAgICAgICAgIGZvciBtIGluIG1lc3NhZ2VzOgogICAgICAgICAgICAgICAgcGFydHMuYXBwZW5kKG0uZ2V0KCJyb2xlIiwgInVzZXIiKSArICI6ICIgKyBtLmdldCgiY29udGVudCIsICIiKSkKICAgICAgICAgICAgcHJvbXB0ID0gIlxuIi5qb2luKHBhcnRzKSArICJcbmFzc2lzdGFudDogIgogICAgICAgIGlucHV0cyA9IHNlbGYubG9jYWxfdG9rZW5pemVyKHByb21wdCwgcmV0dXJuX3RlbnNvcnM9InB0IiwgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbGVuZ3RoPTIwNDgpLnRvKHNlbGYubG9jYWxfbW9kZWwuZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvdXRwdXRzID0gc2VsZi5sb2NhbF9tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9c2VsZi5tYXhfdG9rZW5zLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9bWF4KHNlbGYudGVtcGVyYXR1cmUsIDAuMDEpLAogICAgICAgICAgICAgICAgdG9wX3A9c2VsZi50b3BfcCwKICAgICAgICAgICAgICAgIGRvX3NhbXBsZT1UcnVlLAogICAgICAgICAgICAgICAgcGFkX3Rva2VuX2lkPXNlbGYubG9jYWxfdG9rZW5pemVyLmVvc190b2tlbl9pZCwKICAgICAgICAgICAgKQogICAgICAgIG5ld190b2tlbnMgPSBvdXRwdXRzWzBdW2lucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XQogICAgICAgIHJlc3BvbnNlID0gc2VsZi5sb2NhbF90b2tlbml6ZXIuZGVjb2RlKG5ld190b2tlbnMsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKICAgICAgICByZXR1cm4gcmVzcG9uc2Uuc3RyaXAoKQoKICAgIGRlZiBfYXBpX2NhbGwoc2VsZiwgbWVzc2FnZXMpOgogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBjID0gc2VsZi5jbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgICAgICAgICAgICAgbW9kZWw9c2VsZi5tb2RlbCwgbWVzc2FnZXM9bWVzc2FnZXMsCiAgICAgICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9c2VsZi50ZW1wZXJhdHVyZSwgdG9wX3A9c2VsZi50b3BfcCwKICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPXNlbGYubWF4X3Rva2Vucywgbj0xKQogICAgICAgICAgICAgICAgciA9IGMuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQKICAgICAgICAgICAgICAgIHJldHVybiByLnN0cmlwKCkgaWYgciBlbHNlICIiCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIHdhaXQgPSA1ICogKGF0dGVtcHQgKyAxKQogICAgICAgICAgICAgICAgcHJpbnQoIiAgQVBJIGVycm9yIChhdHRlbXB0ICIgKyBzdHIoYXR0ZW1wdCArIDEpICsgIi81KTogIiArIHN0cihlKVs6MzAwXSArICIuIFdhaXRpbmcgIiArIHN0cih3YWl0KSArICJzLi4uIikKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAod2FpdCkKICAgICAgICByZXR1cm4gIltBUEkgRXJyb3JdIgoKICAgIGRlZiBfc2luZ2xlX2NhbGwoc2VsZiwgbWVzc2FnZXMpOgogICAgICAgIGlmIHNlbGYuX2lzX2xvY2FsOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fbG9jYWxfY2FsbChtZXNzYWdlcykKICAgICAgICByZXR1cm4gc2VsZi5fYXBpX2NhbGwobWVzc2FnZXMpCgogICAgZGVmIF9nZXRfcmVzcG9uc2Uoc2VsZiwgbWVzc2FnZXMsIG51bV9yZXNwb25zZXM9MSk6CiAgICAgICAgaWYgbnVtX3Jlc3BvbnNlcyA+IDE6CiAgICAgICAgICAgIHJlc3BvbnNlcyA9IFtdCiAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKG51bV9yZXNwb25zZXMpOgogICAgICAgICAgICAgICAgcmVzcG9uc2VzLmFwcGVuZChzZWxmLl9zaW5nbGVfY2FsbChtZXNzYWdlcykpCiAgICAgICAgICAgICAgICBpZiBpIDwgbnVtX3Jlc3BvbnNlcyAtIDE6CiAgICAgICAgICAgICAgICAgICAgdGltZS5zbGVlcCgwLjMpCiAgICAgICAgICAgIHJldHVybiByZXNwb25zZXMKICAgICAgICByZXR1cm4gc2VsZi5fc2luZ2xlX2NhbGwobWVzc2FnZXMpCgogICAgZGVmIHF1ZXJ5KHNlbGYsIGFnZW50X25hbWUsIHJvbGVfZGVzYywgaGlzdG9yeV9tZXNzYWdlcywgZ2xvYmFsX3Byb21wdD1Ob25lLAogICAgICAgICAgICAgIHJlcXVlc3RfbXNnPU5vbmUsIG51bV9yZXNwb25zZXM9MSwgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICBpZiBnbG9iYWxfcHJvbXB0OgogICAgICAgICAgICBzeXN0ZW1fcHJvbXB0ID0gZ2xvYmFsX3Byb21wdC5zdHJpcCgpICsgIlxuXG5Zb3VyIG5hbWU6ICIgKyBhZ2VudF9uYW1lICsgIlxuXG5Zb3VyIHJvbGU6ICIgKyByb2xlX2Rlc2MKICAgICAgICBlbHNlOgogICAgICAgICAgICBzeXN0ZW1fcHJvbXB0ID0gcm9sZV9kZXNjCiAgICAgICAgaWYgc2VsZi5tYXhfbGF0ZXN0X21lc3NhZ2VzID4gMCBhbmQgbGVuKGhpc3RvcnlfbWVzc2FnZXMpID4gc2VsZi5tYXhfbGF0ZXN0X21lc3NhZ2VzOgogICAgICAgICAgICBoaXN0b3J5X21lc3NhZ2VzID0gaGlzdG9yeV9tZXNzYWdlc1stc2VsZi5tYXhfbGF0ZXN0X21lc3NhZ2VzOl0KICAgICAgICBhbGxfbWVzc2FnZXMgPSBbKFNZU1RFTV9OQU1FLCBzeXN0ZW1fcHJvbXB0KV0KICAgICAgICBmb3IgbXNnIGluIGhpc3RvcnlfbWVzc2FnZXM6CiAgICAgICAgICAgIGlmIG1zZy5hZ2VudF9uYW1lID09IFNZU1RFTV9OQU1FOgogICAgICAgICAgICAgICAgYWxsX21lc3NhZ2VzLmFwcGVuZCgoU1lTVEVNX05BTUUsIG1zZy5jb250ZW50KSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGFsbF9tZXNzYWdlcy5hcHBlbmQoKG1zZy5hZ2VudF9uYW1lLCBtc2cuY29udGVudCArIEVORF9PRl9NRVNTQUdFKSkKICAgICAgICBpZiByZXF1ZXN0X21zZyBpcyBub3QgTm9uZToKICAgICAgICAgICAgYWxsX21lc3NhZ2VzLmFwcGVuZCgoU1lTVEVNX05BTUUsIHJlcXVlc3RfbXNnLmNvbnRlbnQpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFsbF9tZXNzYWdlcy5hcHBlbmQoKFNZU1RFTV9OQU1FLCAiTm93IHlvdSBzcGVhaywgIiArIGFnZW50X25hbWUgKyAiLiIgKyBFTkRfT0ZfTUVTU0FHRSkpCiAgICAgICAgbWVzc2FnZXMgPSBbXQogICAgICAgIGZvciBpLCBtc2cgaW4gZW51bWVyYXRlKGFsbF9tZXNzYWdlcyk6CiAgICAgICAgICAgIGlmIGkgPT0gMDoKICAgICAgICAgICAgICAgIGFzc2VydCBtc2dbMF0gPT0gU1lTVEVNX05BTUUKICAgICAgICAgICAgICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogbXNnWzFdfSkKICAgICAgICAgICAgZWxpZiBpID09IGxlbihhbGxfbWVzc2FnZXMpIC0gMToKICAgICAgICAgICAgICAgIGFzc2VydCBtc2dbMF0gPT0gU1lTVEVNX05BTUUKICAgICAgICAgICAgICAgIG1lc3NhZ2VzWy0xXVsiY29udGVudCJdID0gbWVzc2FnZXNbLTFdWyJjb250ZW50Il0gKyAiXG5cbiIgKyBtc2dbMV0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGlmIG1zZ1swXSA9PSBhZ2VudF9uYW1lOgogICAgICAgICAgICAgICAgICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAiYXNzaXN0YW50IiwgImNvbnRlbnQiOiBtc2dbMV19KQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBpZiBtZXNzYWdlc1stMV1bInJvbGUiXSA9PSAidXNlciI6CiAgICAgICAgICAgICAgICAgICAgICAgIG1lc3NhZ2VzWy0xXVsiY29udGVudCJdID0gbWVzc2FnZXNbLTFdWyJjb250ZW50Il0gKyAiXG5cblsiICsgbXNnWzBdICsgIl06ICIgKyBtc2dbMV0KICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6ICJbIiArIG1zZ1swXSArICJdOiAiICsgbXNnWzFdfSkKICAgICAgICByZXNwb25zZSA9IHNlbGYuX2dldF9yZXNwb25zZShtZXNzYWdlcywgbnVtX3Jlc3BvbnNlcywgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgIGlmIG51bV9yZXNwb25zZXMgPiAxOgogICAgICAgICAgICByZXNwb25zZSA9IFtyZS5zdWIociJeXHMqXFsuKl06IiwgIiIsIHIpLnN0cmlwKCkgZm9yIHIgaW4gcmVzcG9uc2VdCiAgICAgICAgICAgIHJlc3BvbnNlID0gW3JlLnN1YihyIl5ccyoiICsgcmUuZXNjYXBlKGFnZW50X25hbWUpICsgciJccyo6IiwgIiIsIHIpLnN0cmlwKCkgZm9yIHIgaW4gcmVzcG9uc2VdCiAgICAgICAgICAgIHJlc3BvbnNlID0gW3JlLnN1YihFTkRfT0ZfTUVTU0FHRSArICIkIiwgIiIsIHIpLnN0cmlwKCkgZm9yIHIgaW4gcmVzcG9uc2VdCiAgICAgICAgICAgIGZvciBpZHgsIHIgaW4gZW51bWVyYXRlKHJlc3BvbnNlKToKICAgICAgICAgICAgICAgIGlmIEVORF9PRl9NRVNTQUdFIGluIHI6CiAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2VbaWR4XSA9IHIuc3BsaXQoRU5EX09GX01FU1NBR0UpWzBdLnN0cmlwKCkKICAgICAgICBlbHNlOgogICAgICAgICAgICByZXNwb25zZSA9IHJlLnN1YihyIl5ccypcWy4qXToiLCAiIiwgcmVzcG9uc2UpLnN0cmlwKCkKICAgICAgICAgICAgcmVzcG9uc2UgPSByZS5zdWIociJeXHMqIiArIHJlLmVzY2FwZShhZ2VudF9uYW1lKSArIHIiXHMqOiIsICIiLCByZXNwb25zZSkuc3RyaXAoKQogICAgICAgICAgICByZXNwb25zZSA9IHJlLnN1YihFTkRfT0ZfTUVTU0FHRSArICIkIiwgIiIsIHJlc3BvbnNlKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIEVORF9PRl9NRVNTQUdFIGluIHJlc3BvbnNlOgogICAgICAgICAgICAgICAgcmVzcG9uc2UgPSByZXNwb25zZS5zcGxpdChFTkRfT0ZfTUVTU0FHRSlbMF0uc3RyaXAoKQogICAgICAgIHJldHVybiByZXNwb25zZQo="
patched = base64.b64decode(_patch_b64.encode('ascii')).decode('utf-8')
with open(PATCH_FILE, 'w') as f:
    f.write(patched)
print("✅ VLLMChat patched: Tutor→HF API, Student→LOCAL 4-bit GPU")

print("Applying OOM & 4-Bit Fixes for Phase 1...")

# ── 1. Patch Verifier 4-Bit Quantization AND Hot-Swapping ──
model_utils_path = f"{WORK_DIR}/traver/verifier/model_utils.py"
if os.path.exists(model_utils_path):
    with open(model_utils_path) as f:
        code = f.read()
    
    # 1.a 4-bit config injection
    if "quantization_config" not in code:
        code = code.replace(
            "base_model = AutoModelForCausalLM.from_pretrained(\n        base_model_name_or_path,",
            """bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        quantization_config=bnb_config,"""
        )
    
    # 1.b safely load state_dict to CPU RAM first to avoid loading spikes
    if "map_location='cpu'" not in code:
        code = code.replace("torch.load(trained_verifier_model_path, weights_only=True)", 
                          "torch.load(trained_verifier_model_path, weights_only=True, map_location='cpu')")
                          
    with open(model_utils_path, "w") as f:
        f.write(code)
    print("✅ Verifier OOM Fixes Applied (4-bit + map_location='cpu')")

# ── 2. Patch run_traver.py to HOT-SWAP verifier parts (Fixes VRAM leak) ──
import os
run_traver_path = f"{WORK_DIR}/traver/run_traver.py"
if os.path.exists(run_traver_path):
    with open(run_traver_path) as f:
        code = f.read()
    
    if "Hot-swapping verifier weights" in code:
        print("✅ Verifier Hot-Swapping already enabled.")
    else:
        # We manually find the exact string indices to splice the patch completely layout-agnostic
        start_idx = code.find("if idx > 0:")
        target = "trained_verifier_model_path=verifier_model_path"
        end_idx = code.find(")", code.find(target)) + 1
        
        if start_idx != -1 and end_idx != 0:
            new_block = """if idx == 0:
                    verifier_model, verifer_tokenizer = load_model(
                        base_model_name_or_path=args.verifier_base_model_path,
                        trained_verifier_model_path=verifier_model_path
                    )
                else:
                    import torch, gc
                    print(f"Hot-swapping verifier weights from {verifier_model_path}")
                    state_dict = torch.load(verifier_model_path, map_location="cpu", weights_only=True)
                    verifier_model.load_state_dict(state_dict, strict=False)
                    del state_dict
                    gc.collect(); torch.cuda.empty_cache()"""
            # Splice
            code = code[:start_idx] + new_block + code[end_idx:]
            with open(run_traver_path, "w") as f:
                f.write(code)
            print("✅ Verifier Hot-Swapping Enabled (Fixes VRAM leakage between parts)")
        else:
            print("❌ Warning: Hot-Swapping patch failed. Could not find boundaries.")

print("\n🚀 Ready for Phase 1! Make sure to fully restart your runtime if you hit an OOM previously.")

---
## 5. Phase 1: Run Baseline TRAVER

Runs the standard Traver pipeline with the pre-trained verifier.
Progress is checkpointed — safe to restart if session disconnects.

In [ ]:
import subprocess, os, sys

os.makedirs(f"{WORK_DIR}/traver/utils", exist_ok=True)
with open(f"{WORK_DIR}/traver/__init__.py", "w") as f: pass
os.makedirs(f"{WORK_DIR}/verifier", exist_ok=True)
with open(f"{WORK_DIR}/verifier/__init__.py", "w") as f: pass
with open(f"{WORK_DIR}/traver/utils/__init__.py", "w") as f:
    f.write("from .utils import *\nfrom .make_prompt import *\n")

for level in STUDENT_LEVELS:
    print(f"\n{'='*60}")
    print(f"🎓 Phase 1: TRAVER - {level}")
    print(f"{'='*60}")
    sys.stdout.flush()

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
    # Force python to flush standard output
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_TOKEN"] = HF_TOKEN

    cmd = [
        "python", f"{WORK_DIR}/traver/run_traver.py",
        "--tutor_setting", "traver",
        "--namespace_file", f"{WORK_DIR}/prompt/namespaces.json",
        "--prompt_element_file", f"{WORK_DIR}/prompt/prompt_elements_final.jsonl",
        "--output_dir", f"{WORK_DIR}/output/dialogue",
        "--verifier_base_model_path", MODEL_DIR + "/Mistral-7B-v0.1",
        "--verifier_model_dir", MODEL_DIR + "/Verifier-7B",
        "--tutor_model_name_or_path", TUTOR_MODEL_ID,
        "--tutor_num_responses", str(TUTOR_NUM_RESPONSES),
        "--student_model_name_or_path", STUDENT_MODEL_ID,
        "--student_setting", level,
        "--vllm_api_key", HF_TOKEN,
        "--vllm_endpoint_tutor", HF_API_BASE,
        "--vllm_endpoint_student", "local",
        "--show_description", "false",
        "--show_message", "true",
    ]

    process = subprocess.Popen(cmd, env=env, cwd=WORK_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
        sys.stdout.flush()
        
    process.stdout.close()
    returncode = process.wait()

    if returncode != 0:
        print(f"⚠️ {level} FAILED (exit code {returncode})")
    sys.stdout.flush()

print("\n✅ Phase 1 complete! Dialogues → output/dialogue/traver/")

---
## 6. Phase 2: Run McMiner on Student Code

Extract student code from Phase 1 dialogues, then use McMiner (via Gemini API) to identify misconceptions.

In [ ]:
import json, glob, re

def extract_student_code(dialogue_dir):
    """Extract code snippets from student messages in TRAVER dialogues."""
    samples = []
    for f_path in glob.glob(f"{dialogue_dir}/**/simulated_dialogs.jsonl", recursive=True):
        print(f"📖 {f_path}")
        with open(f_path) as f:
            for line in f:
                data = json.loads(line.strip())
                ns = data.get("namespace", "")
                for turn_idx, turn in enumerate(data.get("conversation", [])):
                    if "student" not in turn:
                        continue
                    msg = turn["student"]
                    blocks = re.findall(r"```(?:python)?\s*(.*?)```", msg, re.DOTALL)
                    if not blocks and any(k in msg for k in ["def ", "class ", "import ", "return "]):
                        blocks = [msg]
                    for ci, code_str in enumerate(blocks):
                        code_str = code_str.strip()
                        if len(code_str) > 20:
                            samples.append({"namespace": ns, "turn_index": turn_idx,
                                          "code_index": ci, "student_code": code_str,
                                          "source_file": f_path})
    return samples

code_samples = extract_student_code(f"{DRIVE_DIR}/output/dialogue/traver")
code_path = f"{DRIVE_DIR}/output/mcminer/extracted_student_code.json"
os.makedirs(os.path.dirname(code_path), exist_ok=True)
with open(code_path, "w") as f:
    json.dump(code_samples, f, indent=2)
print(f"\n📊 Extracted {len(code_samples)} code samples")

In [ ]:
import time
import google.generativeai as genai

# Configure Gemini with key from Colab Secrets
genai.configure(api_key=GOOGLE_API_KEY)

MCMINER_PROMPT = """You are an expert programming instructor. Analyze this student code for
programming MISCONCEPTIONS (fundamental misunderstandings, NOT just bugs/typos).

Student's code:
```python
{student_code}
```

If you find a misconception, respond:
<misconception>
<description>Concise description of the misconception</description>
<explanation>What the student believes vs reality</explanation>
<confidence>high/medium/low</confidence>
</misconception>

If no misconception (code is correct or just has typos): <misconception>NONE</misconception>"""

def run_mcminer_gemini(samples):
    """Run McMiner misconception detection using Gemini API."""
    model = genai.GenerativeModel("gemini-2.5-flash")
    results = []
    total = len(samples)
    start_all = time.time()
    print(f"\U0001f680 Starting MCMiner on {total} samples...", flush=True)

    for i, s in enumerate(samples):
        t0 = time.time()
        prompt = MCMINER_PROMPT.format(student_code=s["student_code"])
        try:
            resp = model.generate_content(prompt)
            raw = resp.text
            elapsed = time.time() - t0
            detected = "NONE" not in raw
            print(f"  [{i+1}/{total}] \u2713 {elapsed:.1f}s \u2014 {'DETECTED' if detected else 'none'}", flush=True)
        except Exception as e:
            elapsed = time.time() - t0
            print(f"  [{i+1}/{total}] \u26a0\ufe0f {elapsed:.1f}s \u2014 {e}", flush=True)
            raw = "<misconception>NONE</misconception>"
            time.sleep(2)

        desc = re.search(r"<description>(.*?)</description>", raw, re.DOTALL)
        expl = re.search(r"<explanation>(.*?)</explanation>", raw, re.DOTALL)
        conf = re.search(r"<confidence>(.*?)</confidence>", raw, re.DOTALL)
        is_none = "NONE" in raw and not desc

        results.append({
            **s,
            "misconception_detected": not is_none,
            "misconception_description": desc.group(1).strip() if desc else None,
            "misconception_explanation": expl.group(1).strip() if expl else None,
            "confidence": conf.group(1).strip() if conf else None,
            "raw_response": raw
        })

    total_time = time.time() - start_all
    print(f"\n\u2705 Done in {total_time:.1f}s ({total_time/max(total,1):.1f}s/sample)", flush=True)
    return results

mcminer_results = run_mcminer_gemini(code_samples)
out_path = f"{DRIVE_DIR}/output/mcminer/misconception_results.json"
with open(out_path, "w") as f:
    json.dump(mcminer_results, f, indent=2)

detected = sum(1 for r in mcminer_results if r["misconception_detected"])
print(f"\n\U0001f4ca McMiner: {detected}/{len(mcminer_results)} misconceptions ({100*detected/max(len(mcminer_results),1):.1f}%)")

---
## 7. Phase 3: Inject Misconceptions & Re-run TRAVER

Augment the tutor prompt with misconception information from McMiner.

In [ ]:
# Build misconception lookup: namespace -> misconceptions
misconception_lookup = {}
for r in mcminer_results:
    if r["misconception_detected"] and r["misconception_description"]:
        ns = r["namespace"]
        if ns not in misconception_lookup:
            misconception_lookup[ns] = []
        misconception_lookup[ns].append({
            "description": r["misconception_description"],
            "explanation": r["misconception_explanation"],
            "confidence": r["confidence"],
            "turn_index": r["turn_index"]})

lookup_path = f"{DRIVE_DIR}/output/mcminer/misconception_lookup.json"
with open(lookup_path, "w") as f:
    json.dump(misconception_lookup, f, indent=2)

print(f"📊 Lookup: {len(misconception_lookup)} namespaces, "
      f"{sum(len(v) for v in misconception_lookup.values())} misconceptions")

In [ ]:
# Create misconception-aware TRAVER wrapper script
script_content = f'''#!/usr/bin/env python3
"""Wrapper: injects McMiner misconceptions into tutor prompt."""
import json, sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "traver"))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

with open("{lookup_path}", "r") as f:
    MISC_LOOKUP = json.load(f)

from traver.utils.make_prompt import prompt_tutor as orig_prompt_tutor
import traver.utils.make_prompt as mp

def mc_prompt_tutor(d, tokenizer, setting="base", max_code_context=1024):
    prompt = orig_prompt_tutor(d, tokenizer, setting=setting, max_code_context=max_code_context)
    ns = d.get("namespace", "")
    if ns in MISC_LOOKUP and setting == "base":
        txt = "\\n\\n--- STUDENT MISCONCEPTION ALERT ---\\n"
        txt += "Detected misconceptions in this student\'s code:\\n"
        for i, m in enumerate(MISC_LOOKUP[ns], 1):
            txt += f"  {{i}}. {{m[\'description\']}}\\n"
            if m.get("explanation"):
                txt += f"     Context: {{m[\'explanation\']}}\\n"
        txt += ("\\nGuide the student to discover and correct these misconceptions "
               "step-by-step using Socratic questioning. Do NOT give direct answers.\\n"
               "--- END ALERT ---")
        prompt += txt
    return prompt

mp.prompt_tutor = mc_prompt_tutor
from traver.run_traver import parse_args, main
args = parse_args()
print(f"\\n🧠 McMiner-TRAVER: {{len(MISC_LOOKUP)}} namespaces with misconceptions")
main(args)
\'\'\'

script_path = f"{WORK_DIR}/run_traver_mcminer.py"
with open(script_path, "w") as f:
    f.write(script_content)
print(f"✅ Created: {script_path}")

In [ ]:
import subprocess, os, sys

os.makedirs(f"{WORK_DIR}/traver/utils", exist_ok=True)
with open(f"{WORK_DIR}/traver/__init__.py", "w") as f: pass
os.makedirs(f"{WORK_DIR}/verifier", exist_ok=True)
with open(f"{WORK_DIR}/verifier/__init__.py", "w") as f: pass
with open(f"{WORK_DIR}/traver/utils/__init__.py", "w") as f:
    f.write("from .utils import *\nfrom .make_prompt import *\n")

for level in STUDENT_LEVELS:
    print(f"\n{'='*60}")
    print(f"🧠 Phase 3: McMiner-TRAVER - {level}")
    print(f"{'='*60}")
    sys.stdout.flush()

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_TOKEN"] = HF_TOKEN

    cmd = [
        "python", f"{WORK_DIR}/run_traver_mcminer.py",
        "--tutor_setting", "traver",
        "--namespace_file", f"{WORK_DIR}/prompt/namespaces.json",
        "--prompt_element_file", f"{WORK_DIR}/prompt/prompt_elements_final.jsonl",
        "--output_dir", f"{WORK_DIR}/output/dialogue_mcminer",
        "--verifier_base_model_path", MODEL_DIR + "/Mistral-7B-v0.1",
        "--verifier_model_dir", MODEL_DIR + "/Verifier-7B",
        "--tutor_model_name_or_path", TUTOR_MODEL_ID,
        "--tutor_num_responses", str(TUTOR_NUM_RESPONSES),
        "--student_model_name_or_path", STUDENT_MODEL_ID,
        "--student_setting", level,
        "--vllm_api_key", HF_TOKEN,
        "--vllm_endpoint_tutor", HF_API_BASE,
        "--vllm_endpoint_student", "local",
        "--show_description", "false",
        "--show_message", "true",
    ]

    process = subprocess.Popen(cmd, env=env, cwd=WORK_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
        sys.stdout.flush()
        
    process.stdout.close()
    returncode = process.wait()

    if returncode != 0:
        print(f"⚠️ {level} FAILED (exit code {returncode})")
    sys.stdout.flush()

print("\n✅ Phase 3 complete!")

---
## 8. Phase 4: Evaluate & Compare

In [ ]:
def load_dialogues(d):
    all_d = []
    for fp in glob.glob(f"{d}/**/simulated_dialogs.jsonl", recursive=True):
        with open(fp) as f:
            for line in f:
                data = json.loads(line.strip())
                for p in fp.split("/"):
                    if p in ["low_level", "med_level", "high_level"]:
                        data["student_level"] = p
                        break
                all_d.append(data)
    return all_d

def show_stats(dialogues, label):
    levels = {}
    for d in dialogues:
        lv = d.get("student_level", "unknown")
        if lv not in levels:
            levels[lv] = {"count": 0, "turns": 0}
        levels[lv]["count"] += 1
        levels[lv]["turns"] += len(d.get("conversation", []))
    total = sum(l["turns"] for l in levels.values())
    n = max(len(dialogues), 1)
    print(f"\n📊 {label}:")
    print(f"  Total: {len(dialogues)} dialogues, {total/n:.1f} avg turns")
    for lv, ld in sorted(levels.items()):
        print(f"  {lv}: {ld['count']} dialogues, {ld['turns']/max(ld['count'],1):.1f} avg turns")

print("="*60)
print("📈 EVALUATION COMPARISON")
print("="*60)
baseline = load_dialogues(f"{DRIVE_DIR}/output/dialogue/traver")
mcminer = load_dialogues(f"{DRIVE_DIR}/output/dialogue_mcminer/traver")
show_stats(baseline, "Baseline TRAVER")
show_stats(mcminer, "McMiner-TRAVER")

In [ ]:
# Full evaluation instructions
print("⚠️  Full pass@k evaluation requires the EvoCodeBench execution environment.")
print("   The dialogue statistics above provide a quick comparison.")
print()
print("For full evaluation, run these scripts:")
print(f"  1. cd {WORK_DIR}")
print(f"  2. bash scripts/run/run_pretest.sh")
print(f"  3. bash scripts/run/run_code_gen.sh")
print(f"  4. bash scripts/run/run_coding_test.sh")
print(f"  5. python scripts/eval/eval_TOR.py")

---
## 📋 Summary

| Phase | Description | Output |
|-------|-------------|--------|
| **1** | Baseline TRAVER dialogues | `output/dialogue/traver/` |
| **2** | McMiner misconception analysis | `output/mcminer/` |
| **3** | McMiner-aware TRAVER dialogues | `output/dialogue_mcminer/traver/` |
| **4** | Evaluation comparison | Stats above |

All outputs saved to Google Drive: `/content/drive/MyDrive/Coding-Tutor-Colab/output/`

**Session-safe**: Progress is checkpointed. Re-run cells if disconnected.